# Knee MRI Multi-Label Classification: Exploratory Data Analysis (EDA)

This notebook provides a complete exploratory workflow for the Knee MRI dataset:
- `data/sample/train.csv`: Study metadata, radiology reports, and 12 binary target pathology conditions.
- `data/sample/train_series.csv`: MRI series acquisition parameters (planes, fluid sensitivity flags).
- `data/sample/*.dcm`: Sample DICOM files.

In [ ]:
# ------------------------------------------------------------------------------
# Step 1: Environment Setup & Target Definitions
# ------------------------------------------------------------------------------
import os
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom

# The 12 multi-label binary targets
TARGET_COLUMNS = [
    "ACL",              # Anterior Cruciate Ligament tear / pathology
    "MCL",              # Medial Collateral Ligament tear / pathology
    "Medial Meniscus",  # Medial Meniscal tear / pathology
    "Lateral Meniscus", # Lateral Meniscal tear / pathology
    "Medial OA",        # Medial Compartment Osteoarthritis
    "Lateral OA",       # Lateral Compartment Osteoarthritis
    "PF OA",            # Patellofemoral Osteoarthritis
    "Effusion",         # Joint Effusion
    "Synovitis",        # Synovitis / Synovial Thickening
    "Baker's",          # Baker's Cyst
    "Contusion",        # Bone Marrow Edema / Contusion
    "Fracture",         # Bone Fracture
]

DATA_DIR = Path("data/sample")
train_df = pd.read_csv(DATA_DIR / "train.csv")
series_df = pd.read_csv(DATA_DIR / "train_series.csv")

print(f"train.csv shape:        {train_df.shape}")
print(f"train_series.csv shape: {series_df.shape}")

## 2. Target Class & Imbalance Distribution
Extracting positive/negative prevalence, imbalance ratios, and co-occurrence across the labeled subset.

In [ ]:
# Filter labeled studies (N=58) from unlabeled studies (N=4,349)
labeled_df = train_df[train_df[TARGET_COLUMNS].notna().any(axis=1)].copy()
unlabeled_df = train_df[train_df[TARGET_COLUMNS].isna().all(axis=1)].copy()

print(f"Ground-truth labeled studies: {len(labeled_df)} ({len(labeled_df)/len(train_df)*100:.2f}%)")
print(f"Unlabeled studies (reports only): {len(unlabeled_df)} ({len(unlabeled_df)/len(train_df)*100:.2f}%)")

# Compute class statistics
stats = []
for col in TARGET_COLUMNS:
    pos = int(labeled_df[col].sum())
    neg = int((labeled_df[col] == 0).sum())
    stats.append({
        "Target Class": col,
        "Positive (1)": pos,
        "Negative (0)": neg,
        "Pos Prevalence (%)": f"{(pos / len(labeled_df)) * 100:.1f}%",
        "Imbalance (Neg:Pos)": f"{neg / pos:.2f}:1" if pos > 0 else "N/A"
    })

pd.DataFrame(stats)

### Co-occurrence Matrix (Pearson Correlation)

In [ ]:
# Display correlation across target pathologies
labeled_df[TARGET_COLUMNS].corr().round(2)

In [ ]:
# Create a heatmap based on the Co-occurrence Matrix for me, make sure to use only two color (blue: negative, red: positive). With ggplot style graph.
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# Apply ggplot style
plt.style.use("ggplot")

# Compute Pearson correlation co-occurrence matrix
corr = labeled_df[TARGET_COLUMNS].corr()

# Define two-color diverging colormap (blue: negative, red: positive)
cmap = LinearSegmentedColormap.from_list("blue_red", ["#2166ac", "#f7f7f7", "#b2182b"])

# Plot co-occurrence heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap=cmap,
    vmin=-1.0,
    vmax=1.0,
    center=0.0,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Pearson Correlation", "shrink": 0.8},
)
plt.title("Target Pathology Co-occurrence Heatmap", fontsize=14, pad=12, fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 3. MRI Series & Sequence Analysis
Analyzing anatomical planes (Sagittal, Coronal, Axial) and series counts per study.

In [ ]:
# Anatomical plane breakdown
print("Anatomical Plane Distribution:")
print(series_df["Anatomical_Plane"].value_counts(normalize=True) * 100)

# Series per study stats
series_per_study = series_df.groupby("StudyInstanceUID").size()
print("\nSeries per Study Description:")
print(series_per_study.describe())

# Verify Fluid_Sensitive vs Fat_Suppression
print("\nAre Fluid_Sensitive and Fat_Suppression identical?", (series_df["Fluid_Sensitive"] == series_df["Fat_Suppression"]).all())

## 4. Multilingual Radiology Reports
Inspecting report lengths and language distributions.

In [ ]:
train_df["char_count"] = train_df["Report"].astype(str).str.len()
train_df["word_count"] = train_df["Report"].astype(str).apply(lambda x: len(x.split()))
print(train_df[["char_count", "word_count"]].describe())

# Sample preview
print("\nSample Report:\n", train_df.iloc[0]["Report"][:400])

## 5. Sample DICOM Inspection

In [ ]:
for f in glob.glob("data/sample/*.dcm"):
    ds = pydicom.dcmread(f)
    print(f"File: {os.path.basename(f)}")
    print(f"  Modality:        {ds.get('Modality')}")
    print(f"  Matrix:          {ds.Rows}x{ds.Columns}")
    print(f"  Pixel Spacing:   {ds.get('PixelSpacing')}")
    print(f"  Slice Thickness: {ds.get('SliceThickness')} mm\n")